In [2]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [15]:
TARGET = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate" ]
TEXT = "comment_text"
ID = "id"

SEED = 42
n_estimator = 500

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [7]:
import pandas as pd
from ydata_profiling import ProfileReport

# Membaca data CSV

# Membuat ringkasan data otomatis
profile = ProfileReport(train, title="Ringkasan Dataset CSV")
profile.to_file("ringkasan_data_train.html")


C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hajar\AppData\Local\Temp\ipykernel_49356\1049156430.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 91.27it/s]


In [8]:
# drop kolom
drop_col = ['id']
train_clean = train.drop(drop_col, axis=1)
test_clean = test.drop(drop_col, axis=1)

In [9]:
# Basic Cleaning

import re
from nltk.corpus import stopwords
import nltk
import matplotlib.pyplot as plt

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hajar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

In [12]:
train_clean['clean_text'] = train_clean[TEXT].apply(clean_text)
test_clean['clean_text'] = test_clean[TEXT].apply(clean_text)

In [13]:
print(train_clean['clean_text'].describe())

count     159571
unique    157855
top             
freq          47
Name: clean_text, dtype: object


In [19]:
#!pip install iterative-stratification

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

X = train_clean['clean_text']
y = train_clean[TARGET]

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED
)

train_idx, val_idx = next(msss.split(X, y))

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

In [20]:
#Embeddings

vectorizer = TfidfVectorizer(max_features=500)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vect = vectorizer.transform(X_val)

In [24]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced'
    )
)

lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_val_vect)
print("Logistic Regression")
print(classification_report(y_val, y_pred_lr))

Logistic Regression
              precision    recall  f1-score   support

           0       0.34      0.81      0.48      3059
           1       0.11      0.86      0.20       319
           2       0.33      0.80      0.46      1690
           3       0.03      0.74      0.05        96
           4       0.23      0.83      0.36      1575
           5       0.06      0.74      0.10       281

   micro avg       0.22      0.81      0.34      7020
   macro avg       0.18      0.80      0.27      7020
weighted avg       0.28      0.81      0.41      7020
 samples avg       0.04      0.08      0.05      7020



C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [25]:
# XGBoost
import xgboost as xgb

# Encoding Target KLO targetnya kategorical
# y_train_enc = y_train.map({'negatif' : 0, 'positif': 1})
# y_val_enc = y_val.map({'negatif' : 0, 'positif': 1})

xgb_model = OneVsRestClassifier(
    xgb.XGBClassifier(
        n_estimators=n_estimator,
        max_depth=20 ,
        random_state=SEED,
        n_jobs=-1)
    )

xgb_model.fit(X_train_vec, y_train) #sesuaikan X nya jika di encode

print('XGBoost')
print(classification_report(y_val, xgb_model.predict(X_val_vect)))

XGBoost
              precision    recall  f1-score   support

           0       0.82      0.47      0.60      3059
           1       0.40      0.17      0.23       319
           2       0.86      0.59      0.70      1690
           3       0.19      0.05      0.08        96
           4       0.71      0.44      0.55      1575
           5       0.53      0.18      0.27       281

   micro avg       0.78      0.46      0.58      7020
   macro avg       0.59      0.32      0.40      7020
weighted avg       0.77      0.46      0.57      7020
 samples avg       0.04      0.04      0.04      7020



C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [26]:
#Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=n_estimator,
        max_depth=10 ,
        random_state=SEED)
)

rf.fit(X_train_vec, y_train)
print("Random Forest")
print(classification_report(y_val, rf.predict(X_val_vect)))

Random Forest
              precision    recall  f1-score   support

           0       1.00      0.10      0.19      3059
           1       0.00      0.00      0.00       319
           2       0.95      0.13      0.23      1690
           3       0.00      0.00      0.00        96
           4       0.94      0.06      0.11      1575
           5       0.00      0.00      0.00       281

   micro avg       0.97      0.09      0.17      7020
   macro avg       0.48      0.05      0.09      7020
weighted avg       0.87      0.09      0.16      7020
 samples avg       0.01      0.01      0.01      7020



C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\INAH-MINE\GE

In [29]:
import numpy as np

submission_baseline = pd.read_csv('sample_submission.csv')
submission_baseline.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.5,0.5,0.5,0.5,0.5,0.5
1,0000247867823ef7,0.5,0.5,0.5,0.5,0.5,0.5
2,00013b17ad220c46,0.5,0.5,0.5,0.5,0.5,0.5
3,00017563c3f7919a,0.5,0.5,0.5,0.5,0.5,0.5
4,00017695ad8997eb,0.5,0.5,0.5,0.5,0.5,0.5


In [30]:
submission_baseline['target'] = np.argmax(lr.predict(X_val_vect))